# 03 · Collect resolved Docker issues from GitHub

Turns **closed-as-completed** issues from Docker's public repos into `(problem, resolution)` rows, where the
resolution is the maintainer comment that closed the loop (and whether a linked PR fixed it).

**Why GraphQL:** one request returns 50 issues *and* their latest comments (~5 rate-limit points), so tens of
thousands of issues fit in the hourly budget. The REST API would need one request per issue.

## Before you run
1. Create a token: GitHub → Settings → Developer settings → *Fine-grained tokens* → **Public repositories (read-only)**, no extra permissions.
2. In Colab click the key icon (left sidebar) → add a secret named `GITHUB_TOKEN` with that value → enable notebook access.
   (If the secret is missing you will be asked to paste it; the input is hidden and never stored in the notebook.)

**Resumable:** progress per repo is checkpointed to Drive. If Colab disconnects, run again and it picks up at the last page.

Output: `raw_sources/github/<owner>__<repo>.jsonl`, consumed by notebook 05. Issue text is user content under GitHub's Terms of Service; each row keeps its `url` for attribution.

In [ ]:
# ---- Config ---------------------------------------------------------------------------
# (owner, repo, max issues to keep). Counts of closed-as-completed issues at time of writing:
# moby ~18.7k, for-win ~13.4k, compose ~7.9k, for-mac ~6.8k, hub-feedback ~2.4k, buildx ~1.2k, cli ~0.9k, for-linux ~0.8k
REPOS = [
    ("moby", "moby", 3000),
    ("docker", "for-win", 2500),
    ("docker", "compose", 2500),
    ("docker", "for-mac", 2500),
    ("docker", "hub-feedback", 900),
    ("docker", "buildx", 800),
    ("docker", "cli", 600),
    ("docker", "for-linux", 500),
]
MAX_PAGES_PER_REPO = 400      # safety cap on pages scanned per repo (50 issues per page)
MAX_COMMENTS = 60             # skip mega-threads: the answer is rarely a single clear comment
SKIP_LABEL_WORDS = ("feature", "enhancement", "proposal", "roadmap")   # not troubleshooting
USE_DRIVE = True

!pip install -q requests

In [ ]:
import getpass
import json
import os
import re
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import requests

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_ROOT = Path("/content/drive/MyDrive/customer-support-triage/tickets")
    except ImportError:
        OUT_ROOT = Path("tickets")
else:
    OUT_ROOT = Path("tickets")

GH_DIR = OUT_ROOT / "raw_sources" / "github"
GH_DIR.mkdir(parents=True, exist_ok=True)


def get_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)


GITHUB_TOKEN = get_secret("GITHUB_TOKEN") or getpass.getpass("GitHub token (input hidden): ")
print("Writing to:", OUT_ROOT.resolve())

## 1 · GraphQL query and client

In [ ]:
QUERY = '''
query($owner: String!, $name: String!, $after: String, $pageSize: Int!) {
  rateLimit { cost remaining resetAt }
  repository(owner: $owner, name: $name) {
    issues(first: $pageSize, after: $after, states: CLOSED, orderBy: {field: CREATED_AT, direction: DESC}) {
      pageInfo { hasNextPage endCursor }
      nodes {
        number title url body createdAt closedAt stateReason
        author { login }
        reactions { totalCount }
        labels(first: 10) { nodes { name } }
        closedByPullRequestsReferences(first: 1) { nodes { url } }
        comments(last: 10) {
          totalCount
          nodes { authorAssociation body createdAt isMinimized author { login __typename } }
        }
      }
    }
  }
}
'''

session = requests.Session()
session.headers.update({"Authorization": f"Bearer {GITHUB_TOKEN}"})


def graphql(variables):
    for attempt in range(6):
        resp = session.post("https://api.github.com/graphql", json={"query": QUERY, "variables": variables}, timeout=90)
        if resp.status_code in (502, 503, 504) or (resp.status_code == 403 and "rate limit" in resp.text.lower()):
            time.sleep(min(90, 5 * 2 ** attempt))
            continue
        if resp.status_code == 401:
            raise SystemExit("GitHub rejected the token (401). Check the GITHUB_TOKEN secret.")
        resp.raise_for_status()
        payload = resp.json()
        if payload.get("errors") and not payload.get("data"):
            raise RuntimeError(payload["errors"])
        return payload["data"]
    raise RuntimeError("GitHub GraphQL kept failing")


def wait_for_rate_limit(rate):
    if rate["remaining"] < 100:
        reset = datetime.fromisoformat(rate["resetAt"].replace("Z", "+00:00"))
        pause = max(5, (reset - datetime.now(timezone.utc)).total_seconds() + 5)
        print(f"  rate limit low ({rate['remaining']}); sleeping {int(pause)}s")
        time.sleep(pause)

## 2 · Picking the resolution

A resolution is the last substantive comment from someone with **owner / member / collaborator** status, written
before (or within a day of) the issue closing. Bots, minimised (spam) comments, and boilerplate such as
"closing due to inactivity" or "duplicate of #123" do not count. If no maintainer answered, a substantive comment from
someone other than the reporter is used and labelled `github_community_resolved` (lower trust).

In [ ]:
MAINTAINER_ROLES = {"OWNER", "MEMBER", "COLLABORATOR"}
BOILERPLATE_RE = re.compile(
    r"(?i)(closing( this)?( issue)?|closed due to|stale|inactiv\w*|duplicate of|dupe of|"
    r"please (open|file|create) a new issue|marking as|this issue has been automatically|"
    r"\+1|me too|same (here|issue)|any update|\bbump\b|thanks?[ !.]*$)")
URL_RE = re.compile(r"https?://\S+")
HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.S)


def parse_time(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def is_bot(author):
    return author is None or author.get("__typename") == "Bot" or author["login"].endswith("[bot]")


def substantive(text):
    core = URL_RE.sub("", BOILERPLATE_RE.sub("", text or ""))
    return len(core.strip()) >= 40


def pick_resolution(issue):
    cutoff = parse_time(issue["closedAt"]) + timedelta(days=1)
    reporter = (issue["author"] or {}).get("login")
    usable = [c for c in issue["comments"]["nodes"]
              if c and not c["isMinimized"] and not is_bot(c["author"])
              and parse_time(c["createdAt"]) <= cutoff and substantive(c["body"])]

    maintainers = [c for c in usable if c["authorAssociation"] in MAINTAINER_ROLES and c["author"]["login"] != reporter]
    if maintainers:
        return maintainers[-1]["body"], "maintainer_comment", "github_maintainer_resolved"
    others = [c for c in usable if c["author"]["login"] != reporter]
    if others:
        return others[-1]["body"], "community_comment", "github_community_resolved"
    return None, None, None


def issue_to_row(owner, name, issue):
    if issue["stateReason"] not in ("COMPLETED", None):
        return None
    total = issue["comments"]["totalCount"]
    labels = [l["name"] for l in issue["labels"]["nodes"]]
    if not 1 <= total <= MAX_COMMENTS or any(w in l.lower() for l in labels for w in SKIP_LABEL_WORDS):
        return None

    problem = HTML_COMMENT_RE.sub("", issue["body"] or "").strip()
    if len(problem.split()) < 15:
        return None
    resolution, kind, source_type = pick_resolution(issue)
    if not resolution:
        return None

    fixed_by = issue["closedByPullRequestsReferences"]["nodes"]
    return {
        "source_id": f"github:{owner}/{name}#{issue['number']}",
        "source": f"github/{owner}/{name}",
        "source_type": source_type,
        "url": issue["url"],
        "license": "GitHub user content (cite by URL)",
        "created_at": issue["createdAt"],
        "title": issue["title"].strip(),
        "problem": problem,
        "resolution": resolution.strip(),
        "resolution_kind": kind,
        "problem_score": issue["reactions"]["totalCount"],
        "resolution_score": None,
        "tags": labels,
        "extra": {"comments_total": total, "state_reason": issue["stateReason"],
                  "closed_at": issue["closedAt"], "closed_by_pr": fixed_by[0]["url"] if fixed_by else None},
    }

## 3 · Collect (resumable)

In [ ]:
def collect_repo(owner, name, max_kept):
    rows_path = GH_DIR / f"{owner}__{name}.jsonl"
    state_path = GH_DIR / f"{owner}__{name}.state.json"
    state = json.loads(state_path.read_text()) if state_path.exists() else {"after": None, "pages": 0, "kept": 0, "done": False}
    if state["done"] or state["kept"] >= max_kept:
        print(f"{owner}/{name}: already complete ({state['kept']} rows)")
        return state

    with rows_path.open("a", encoding="utf-8") as out:
        while state["pages"] < MAX_PAGES_PER_REPO and state["kept"] < max_kept:
            data = graphql({"owner": owner, "name": name, "after": state["after"], "pageSize": 50})
            page = data["repository"]["issues"]
            for issue in page["nodes"]:
                row = issue_to_row(owner, name, issue)
                if row:
                    out.write(json.dumps(row, ensure_ascii=False) + "\n")
                    state["kept"] += 1
            state["pages"] += 1
            state["after"] = page["pageInfo"]["endCursor"]
            state["done"] = not page["pageInfo"]["hasNextPage"]
            state_path.write_text(json.dumps(state))
            if state["pages"] % 10 == 0:
                print(f"  {owner}/{name}: page {state['pages']}, kept {state['kept']}")
            if state["done"]:
                break
            wait_for_rate_limit(data["rateLimit"])
    print(f"{owner}/{name}: kept {state['kept']} after {state['pages']} pages")
    return state


summary = {f"{o}/{n}": collect_repo(o, n, cap) for o, n, cap in REPOS}
(GH_DIR / "collect_report.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

## 4 · Spot-check

In [ ]:
import random

for o, n, _ in REPOS:
    path = GH_DIR / f"{o}__{n}.jsonl"
    if not path.exists():
        continue
    rows = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines()]
    kinds = {}
    for r in rows:
        kinds[r["source_type"]] = kinds.get(r["source_type"], 0) + 1
    print(f"\n=== {o}/{n}: {len(rows)} rows {kinds}")
    for row in random.sample(rows, min(2, len(rows))):
        print(f"[{row['created_at'][:10]}] {row['title'][:100]}")
        print("    ->", row["resolution"][:160].replace("\n", " "))